# Kibana Runtime Fields: Deriving Day of Week from Timestamps

## Overview

Runtime fields in Kibana let you compute new values from existing data **at query time**, without modifying your index or reindexing documents. This is powerful when your raw data contains the information you need — but not in the shape you want to analyze.

This guide walks through a practical example: deriving the **day of the week** from a timestamp field to identify peak traffic patterns in web server logs.

---

## Why Runtime Fields?

Your Elasticsearch index stores a `@timestamp` field on every document, but it doesn't store a `day_of_week` field. You have two options:

| Approach | When to use |
|---|---|
| **Runtime field** | Exploratory analysis, low-to-medium data volumes, avoiding reindexing |
| **Indexed field** | High-volume data, frequently queried field, production dashboards at scale |

Runtime fields are computed on the fly during each query. This makes them flexible and fast to set up, but they carry a **performance cost on large datasets** since the computation happens per document, per query.

---

## Step-by-Step Guide

### Step 1 – Open Discover with the Right Data View

1. In Kibana, navigate to **☰ > Analytics > Discover**.
2. Confirm the **Kibana Sample Data Logs** data view is selected in the top-left dropdown.
3. Set the time range to **Last 7 days** using the date picker in the top-right.

> **What you're looking at:** Each row in Discover is a document from your Elasticsearch index. The left panel lists all available fields — both indexed fields stored in the index and any runtime fields you've defined.

---

### Step 2 – Identify What Field to Derive From

Scan the available fields list on the left. You'll notice there is no `day_of_week` field.

The field to derive from is **`@timestamp`** — every log document has one, and it encodes a full point in time, from which day of week can be mathematically derived.

> **Core insight:** You don't need the data to *already contain* the value you want. As long as the raw data has a field that *implies* the value — a timestamp, a URL, a full name — a runtime field can extract or compute it on the fly.

---

### Step 3 – Open the Create Field Panel

Scroll to the bottom of the available fields list in the left sidebar and click **"Add a field"**.

This opens the **Create field** panel, which is Kibana's UI for defining runtime fields tied to the current data view.

---

### Step 4 – Configure the Runtime Field

Fill in the panel with the following:

#### Name
```
day_of_week
```

#### Type
Keep **Keyword** selected. Since the output will be a day name like `"Monday"` or `"Friday"`, a keyword type is appropriate — it's a discrete string value, not a number or date.

#### Script

Toggle **"Set value"** to enable the Painless script editor, then enter:

```painless
emit(doc['@timestamp'].value.dayOfWeekEnum.getDisplayName(TextStyle.FULL, Locale.ROOT))
```

#### Breaking Down the Script

| Part | What it does |
|---|---|
| `doc['@timestamp']` | Accesses the `@timestamp` field on the current document via the doc-values API |
| `.value` | Retrieves the actual `ZonedDateTime` object from Elasticsearch |
| `.dayOfWeekEnum` | Extracts a Java `DayOfWeek` enum (e.g., `MONDAY`, `TUESDAY`) |
| `.getDisplayName(TextStyle.FULL, Locale.ROOT)` | Converts the enum to a human-readable full name (`"Monday"`, `"Tuesday"`, etc.) using locale-neutral formatting |
| `emit(...)` | **Required in all runtime field scripts** — this is how the script returns its computed value back to the query engine |

> **On `emit`:** Unlike regular Painless scripts, runtime field scripts don't use `return`. The `emit()` function explicitly pushes a value into the runtime field output. You can even call `emit()` multiple times to produce a multi-valued field.

#### Preview Panel

The right side of the panel shows a live preview of the `day_of_week` value for a randomly selected document. Use the **left/right arrows** to cycle through documents and verify the script produces correct output before saving.

---

### Step 5 – Save and Use the Field

Click **Save**. The `day_of_week` field now appears in the available fields list alongside all indexed fields.

You can immediately use it for:

- **Filtering** — show only requests from `"Saturday"` or `"Sunday"`
- **Aggregations** — count documents grouped by day of week
- **Visualizations** — build a bar chart in Lens showing request volume per day
- **Dashboards** — surface busiest days across any time range

> **Scope note:** Runtime fields defined this way are saved to the **data view** (formerly index pattern), not the index itself. They are available across all Kibana apps that use that data view — Discover, Lens, Maps, and Alerting.

---

## Performance Considerations

Because runtime fields are evaluated at query time rather than stored, keep the following in mind:

- **Small/medium datasets:** Runtime fields are typically fast enough for interactive exploration.
- **Large datasets (billions of docs):** Query latency can increase significantly. Consider materializing the field in your index mapping using an ingest pipeline or updating your data stream's component template.
- **Repeated production use:** If a runtime field becomes load-bearing in dashboards or alerts, promote it to an indexed field to avoid repeated compute overhead.

---

## Using the Runtime Field in a Visualization

Once saved, the `day_of_week` runtime field is available across every Kibana app that uses the same data view — Discover, Lens, Maps, and Alerting — without any extra setup.

### Step 1 – Open the Field Stats / Quick Visualization

In the Discover fields list, hover over `day_of_week` and click **"Top 5 values of day_of_week"** on the right side. This opens a quick visualization panel showing the most frequent day values in your current time range.

### Step 2 – Expand to All 7 Days

The default shows only the top 5 values. Change **Number of values** to `7`.

> **Why 7?** There are exactly 7 days in a week. Capping at 5 risks hiding lower-traffic days (often weekends), which skews your picture of weekly traffic distribution. Setting it to 7 guarantees every day is represented — even if some have relatively few requests.

### Step 3 – Save to Dashboard

Click **"Save and return"** to add the visualization to your dashboard. Then click **Save** on the dashboard itself to persist the layout.

> **Scope reminder:** The runtime field is defined at the data view level, not per-visualization. Any future charts or dashboards using the same data view automatically have access to `day_of_week` without recreating it.

---

## Summary

| Concept | Detail |
|---|---|
| **Runtime field** | A field computed at query time from a Painless script |
| **Source field used** | `@timestamp` |
| **Output field** | `day_of_week` (Keyword) |
| **Script language** | Painless (Elasticsearch's built-in scripting language) |
| **Key method** | `emit()` — required to return values from runtime scripts |
| **Performance trade-off** | Flexible and reindexing-free, but adds query-time compute cost |

Runtime fields are one of the most practical tools for **schema-on-read analysis** in Kibana — letting you reshape and enrich your data view without touching the underlying index.
